Solver Demos


In [1]:
import numpy as np
from anser import *
import os
os.chdir("/home/patrick/ansermodelling")
from models.eval import report_error_stats
from models import *
from models.train import pose_errors
import torch
import matplotlib.pyplot as plt


In [2]:
test_set = np.load("data/test_set.npz")
measurements = test_set["xs"][:10000]
poses = test_set["ys"][:10000]

In [3]:
coils_global = build_field_generator(N_turns,l,w,s,z_thick,centres,rotations)
f = lambda x : forward_model(x,coils_global)


In [4]:
lm = LMSolver(f,np.array([0.0,0.0,0.125,0.0,1.0,0.0])) 

In [5]:
%%time
poses_pred_lm, success_lm = lm.solve(measurements)

CPU times: user 7min 51s, sys: 78.9 ms, total: 7min 51s
Wall time: 7min 53s


In [6]:
report_error_stats(poses_pred_lm,poses,success_lm)

Mean pos error: 9.43, mean angle error: 8.95
Median pos error: 1.49e-11, Median angle error: 0.0256
95% pos error : 61.8 95% angle error: 79.5
LM success rate: 0.875
Convergence rate: 0.857
Mean pos error of converged: 7.97e-10, mean angle error of converged: 0.0256 


In [7]:
model = FFNN(input_dim=8, output_dim=6, hidden_dims=[256,256,256])
model.load_state_dict(torch.load("nn_normal.pt"))

<All keys matched successfully>

In [8]:
nn = NNSolver(model)

In [9]:
%%time
poses_pred_nn, success_nn = nn.solve(measurements)

CPU times: user 172 ms, sys: 27.5 ms, total: 199 ms
Wall time: 39.7 ms


In [10]:
report_error_stats(poses_pred_nn,poses,success_nn)

Mean pos error: 8.77, mean angle error: 6.2
Median pos error: 7.02, Median angle error: 3.44
95% pos error : 20.6 95% angle error: 20
LM success rate: 1
Convergence rate: 0.0005
Mean pos error of converged: 0.761, mean angle error of converged: 0.857 


In [11]:
hy = HybridSolver(model,f)

In [12]:
%%time
poses_pred_hy, success_hy = hy.solve(measurements)

CPU times: user 2min 41s, sys: 43.5 ms, total: 2min 41s
Wall time: 2min 41s


In [13]:
report_error_stats(poses_pred_hy,poses,success_hy)

Mean pos error: 1.09, mean angle error: 1.94
Median pos error: 3.86e-12, Median angle error: 0.0256
95% pos error : 1.67 95% angle error: 3.13
LM success rate: 0.959
Convergence rate: 0.944
Mean pos error of converged: 4.84e-10, mean angle error of converged: 0.0256 
